# Soal 2 · Pemrosesan dengan Spark RDD
**Big Data** · notebook ini adalah **driver**, dijalankan di **JupyterLab PC Master** (192.168.1.15).
Executor berjalan di **4 Spark Worker** (4 PC × 4 core = 16 core).

Data: snapshot `wiki_snapshot.jsonl` (222.241 pesan) dan `wiki_snapshot_uji.jsonl` (+100 baris rusak sisipan),
disimpan di path yang **sama** di kelima PC: `C:\BigData\data\snapshot_wiki\`.

Pasangan notebook ini: **Soal2_Master_Monitor.ipynb** (monitor cluster dan task, tab terpisah).

**Bagian ini: Soal 2a.** Sel untuk 2b–2d ditambahkan di bawahnya nanti, dalam SparkSession yang sama.

Spark UI selama notebook hidup: `http://192.168.1.15:4040` → tab **Jobs** (label `2a-...`) dan **Stages**.
Jika muncul *Cannot call methods on a stopped SparkContext*: **Kernel → Restart Kernel**, lalu jalankan dari Sel 1.

## Sel 1. Konfigurasi dan SparkSession ke cluster

In [ ]:
# ===== SEL 1: Konfigurasi dan SparkSession ke cluster =====
import os, sys, socket, time, json
from pathlib import Path
from collections import Counter

from pyspark import SparkContext
from pyspark.sql import SparkSession

# ---------------- ISI SESUAI LINGKUNGAN ----------------
MASTER_IP      = "192.168.1.15"
CORES_MAX      = 16                    # 4 Worker x 4 core (lab-config.json: coresMax)
EXECUTOR_CORES = 4                     # satu executor per PC Worker
EXECUTOR_MEM   = "4g"                  # Worker punya 6g
MIN_PARTISI    = 2 * CORES_MAX         # 32 partisi = 2 task per core
DATA_DIR       = "C:/BigData/data/snapshot_wiki"   # path SAMA di kelima PC
# -------------------------------------------------------

MASTER   = f"spark://{MASTER_IP}:7077"
PATH_ASLI = f"file:///{DATA_DIR}/wiki_snapshot.jsonl"
PATH_UJI  = f"file:///{DATA_DIR}/wiki_snapshot_uji.jsonl"

# IP PC ini (driver) - terdeteksi otomatis seperti di materi RDD-Cluster
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
s.connect((MASTER_IP, 7077)); IP = s.getsockname()[0]; s.close()

EVENT_DIR = Path("spark-events").resolve(); EVENT_DIR.mkdir(exist_ok=True)

old = SparkContext._active_spark_context
if old is not None:
    try: old.stop()
    except Exception: pass
SparkSession._instantiatedSession = None
SparkSession._activeSession = None

spark = (SparkSession.builder
         .appName("Soal2-RDD-Wikipedia")
         .master(MASTER)
         .config("spark.driver.host", IP)
         .config("spark.driver.bindAddress", IP)
         .config("spark.cores.max", str(CORES_MAX))
         .config("spark.executor.cores", str(EXECUTOR_CORES))
         .config("spark.executor.memory", EXECUTOR_MEM)
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .config("spark.ui.port", "4040")
         .config("spark.eventLog.enabled", "true")            # winutils sudah terpasang (Pedoman bag. 7)
         .config("spark.eventLog.dir", EVENT_DIR.as_uri())
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

def tandai(deskripsi):
    """Label untuk Job berikutnya agar mudah dicari di tab Jobs Spark UI."""
    sc.setJobGroup(deskripsi.split()[0], deskripsi)

print("Spark            :", spark.version, "|", sc.master)
print("Driver (PC ini)  :", IP)
print("Application ID   :", sc.applicationId)
print("Spark UI         :", sc.uiWebUrl)
print("defaultParallelism:", sc.defaultParallelism)

## Sel 2. Cek executor dan file di setiap PC
Setiap task melaporkan nama PC tempat ia berjalan dan apakah file snapshot ada di sana. Hasil yang diharapkan: **4 nama PC berbeda**, semuanya menemukan kedua file dengan ukuran yang sama.

In [ ]:
# ===== SEL 2: Executor dan keberadaan file di tiap PC =====
LOKAL_ASLI = DATA_DIR + "/wiki_snapshot.jsonl"
LOKAL_UJI  = DATA_DIR + "/wiki_snapshot_uji.jsonl"

def cek_file(_):
    import os, socket
    uk = lambda p: os.path.getsize(p) if os.path.exists(p) else None
    return (socket.gethostname(), uk(LOKAL_ASLI), uk(LOKAL_UJI))

tandai("2a-cek Cek executor dan file di tiap PC")
hasil = sorted(set(sc.parallelize(range(4 * CORES_MAX), 4 * CORES_MAX).map(cek_file).collect()))
print(f"{'PC (executor)':<22}{'snapshot asli (byte)':>22}{'snapshot uji (byte)':>22}")
for host, a, u in hasil:
    print(f"{host:<22}{str(a):>22}{str(u):>22}")
print(f"{'driver: ' + socket.gethostname():<22}{str(os.path.getsize(LOKAL_ASLI)):>22}{str(os.path.getsize(LOKAL_UJI)):>22}")
print(f"\nJumlah PC executor: {len({h for h, _, _ in hasil})}")
if any(a is None or u is None for _, a, u in hasil):
    print("PERINGATAN: ada PC yang tidak menemukan file -> salin ke", DATA_DIR)

## Sel 3. Fungsi parsing dan accumulator
Setiap baris diubah menjadi **tuple** 12 field. Baris rusak **tidak dibuang diam-diam**: masing-masing menambah accumulator sesuai jenisnya.

| Accumulator | Dihitung bila |
|---|---|
| `acc_kosong` | baris kosong atau hanya spasi |
| `acc_json_rusak` | gagal `json.loads` |
| `acc_field_wajib` | JSON valid tetapi bukan objek, atau `id`/`wiki`/`type`/`title`/`timestamp` hilang atau salah tipe |
| `acc_tanpa_comment` | baris valid tetapi `comment` kosong (pesan producer v1.0 atau suntingan tanpa ringkasan) — **bukan** data rusak |

Parsing memakai `flatMap`: baris valid → 1 tuple, baris rusak → 0 tuple. `flatMap` adalah transformasi **narrow**.

In [ ]:
# ===== SEL 3: Konstanta tuple, accumulator, dan fungsi parsing =====
WIKI, ID, TYPE, TITLE, USER, IS_ANON, BOT, MINOR, TS, LEN_OLD, LEN_NEW, COMMENT = range(12)
NAMA_FIELD = ["wiki", "id", "type", "title", "user", "is_anon", "bot", "minor",
              "timestamp", "length_old", "length_new", "comment"]

def buat_accumulator():
    return {k: sc.accumulator(0) for k in ("kosong", "json_rusak", "field_wajib", "tanpa_comment")}

def nilai(acc):
    return {k: a.value for k, a in acc.items()}

def _int(x):
    return isinstance(x, int) and not isinstance(x, bool)

def buat_parser(acc):
    """Kembalikan fungsi untuk flatMap yang memakai accumulator `acc`."""
    a_kosong, a_json, a_field, a_comment = acc["kosong"], acc["json_rusak"], acc["field_wajib"], acc["tanpa_comment"]
    def parse(baris):
        import json
        if not baris.strip():
            a_kosong.add(1); return []
        try:
            e = json.loads(baris)
        except ValueError:
            a_json.add(1); return []
        if not isinstance(e, dict) or not (
                _int(e.get("id")) and _int(e.get("timestamp"))
                and all(isinstance(e.get(k), str) and e.get(k) for k in ("wiki", "type", "title"))):
            a_field.add(1); return []
        komentar = e.get("comment") or ""
        if not komentar:
            a_comment.add(1)
        return [(e["wiki"], e["id"], e["type"], e["title"], e.get("user"),
                 bool(e.get("is_anon", False)), bool(e.get("bot", False)), bool(e.get("minor", False)),
                 e["timestamp"], e.get("length_old"), e.get("length_new"), komentar)]
    return parse

# Nilai pembanding dari notebook ekspor (Sel 3 dan Sel 4)
HARAPAN = {"valid": 222_241, "kosong": 30, "json_rusak": 40, "field_wajib": 30, "tanpa_comment": 46_226}
print("Fungsi parsing siap. Pembanding:", HARAPAN)

## Sel 4. Membaca snapshot menjadi RDD, jumlah partisi, dan `glom()`
`sc.textFile` memecah file berdasarkan **ukuran byte**. `glom().map(len)` mengubah tiap partisi menjadi list lalu hanya mengirim **panjangnya** ke driver, bukan isi datanya.

In [ ]:
# ===== SEL 4: textFile, jumlah partisi, sebaran elemen per partisi =====
import statistics as st

def sebaran(rdd, label):
    tandai(f"2a-glom glom() {label}")
    n = rdd.glom().map(len).collect()
    print(f"[{label}] partisi={len(n)} | total={sum(n):,} | min={min(n):,} | maks={max(n):,} | "
          f"rata2={st.mean(n):,.0f} | sd={st.pstdev(n):,.0f} | maks/min={max(n) / max(min(n), 1):.3f}")
    return n

mentah_uji = sc.textFile(PATH_UJI, minPartitions=MIN_PARTISI).setName("mentah_uji")
print("Jumlah partisi RDD mentah:", mentah_uji.getNumPartitions())
glom_mentah = sebaran(mentah_uji, "mentah uji (baris teks)")
print("Elemen per partisi:", glom_mentah)

## Sel 5. Parsing file uji: accumulator harus menangkap tepat 30 / 40 / 30
Accumulator dibaca **setelah action pertama** (`count`). Nilainya dibandingkan dengan kunci jawaban sisipan.

In [ ]:
# ===== SEL 5: Parsing snapshot uji + accumulator =====
acc_uji = buat_accumulator()
parsed_uji = mentah_uji.flatMap(buat_parser(acc_uji)).setName("parsed_uji")

tandai("2a-parse Parsing snapshot uji (count)")
t0 = time.time()
n_valid = parsed_uji.count()
dt = time.time() - t0
hasil_acc = nilai(acc_uji)

print(f"Baris di file uji : {sum(glom_mentah):,}")
print(f"Tuple valid       : {n_valid:,}   (harapan {HARAPAN['valid']:,})")
print(f"Waktu count       : {dt:.2f} dtk\n")
print(f"{'Accumulator':<16}{'Hasil':>10}{'Harapan':>10}  Status")
for k in ("kosong", "json_rusak", "field_wajib", "tanpa_comment"):
    print(f"{k:<16}{hasil_acc[k]:>10,}{HARAPAN[k]:>10,}  {'COCOK' if hasil_acc[k] == HARAPAN[k] else 'BEDA'}")
rusak = hasil_acc["kosong"] + hasil_acc["json_rusak"] + hasil_acc["field_wajib"]
print(f"\nCek total: valid {n_valid:,} + rusak {rusak:,} = {n_valid + rusak:,} (baris file {sum(glom_mentah):,})")

print("\nContoh 2 tuple:")
for r in parsed_uji.take(2):
    print(" ", dict(zip(NAMA_FIELD, r)))

glom_parsed = sebaran(parsed_uji, "setelah parsing")
selisih = [a - b for a, b in zip(glom_mentah, glom_parsed)]
print("Baris rusak yang dikeluarkan per partisi:", selisih)
print("Jumlah partisi tetap sama (flatMap = narrow):", parsed_uji.getNumPartitions() == mentah_uji.getNumPartitions())

## Sel 6. Snapshot asli (pembanding)
Data asli tidak berisi baris rusak, sehingga ketiga accumulator rusak harus 0.

In [ ]:
# ===== SEL 6: Parsing snapshot asli =====
acc_asli = buat_accumulator()
mentah_asli = sc.textFile(PATH_ASLI, minPartitions=MIN_PARTISI).setName("mentah_asli")
parsed_asli = mentah_asli.flatMap(buat_parser(acc_asli)).setName("parsed_asli")
tandai("2a-asli Parsing snapshot asli (count)")
n_asli = parsed_asli.count()
print(f"Partisi: {parsed_asli.getNumPartitions()} | tuple valid: {n_asli:,} | accumulator: {nilai(acc_asli)}")

## Sel 7. Jebakan accumulator: terhitung ganda tanpa `cache()`
Spark hanya menjamin accumulator dihitung tepat sekali bila diperbarui di dalam **action**. Di sini accumulator diperbarui di dalam
**transformation** (`flatMap`), jadi setiap kali RDD **dihitung ulang**, accumulator bertambah lagi.
Dengan `cache()`, RDD hasil parsing disimpan di memori executor sehingga action kedua tidak menjalankan `flatMap` lagi.

In [ ]:
# ===== SEL 7: Accumulator tanpa cache vs dengan cache =====
def dua_action(rdd, acc, label):
    hasil = []
    for i in (1, 2):
        tandai(f"2a-acc {label} count #{i}")
        t = time.time(); rdd.count(); d = time.time() - t
        hasil.append((i, d, acc["json_rusak"].value, acc["tanpa_comment"].value))
        print(f"{label:<14} count #{i}: {d:6.2f} dtk | acc_json_rusak={hasil[-1][2]:>4} | acc_tanpa_comment={hasil[-1][3]:>7,}")
    return hasil

acc_a = buat_accumulator()
tanpa_cache = mentah_uji.flatMap(buat_parser(acc_a)).setName("parsed_tanpa_cache")
r1 = dua_action(tanpa_cache, acc_a, "TANPA cache")

acc_b = buat_accumulator()
dengan_cache = mentah_uji.flatMap(buat_parser(acc_b)).setName("parsed_dengan_cache").cache()
r2 = dua_action(dengan_cache, acc_b, "DENGAN cache")

print("\nKesimpulan:")
print(f"- Tanpa cache : accumulator naik dari {r1[0][2]} ke {r1[1][2]} (flatMap dijalankan ulang -> terhitung ganda)")
print(f"- Dengan cache: accumulator tetap {r2[1][2]} (count #2 membaca cache, flatMap tidak dijalankan lagi)")
print("- Jadi nilai accumulator dari transformation hanya dibaca setelah action PERTAMA, atau RDD-nya di-cache.")
dengan_cache.unpersist()       # dilepas agar pengukuran cache di Soal 2d dimulai dari keadaan bersih

## Sel 8. Ringkasan Soal 2a

In [ ]:
# ===== SEL 8: Ringkasan untuk laporan =====
print("===== RINGKASAN SOAL 2a =====")
print(f"Sumber                  : {PATH_UJI}")
print(f"Executor                : {len({h for h, _, _ in hasil})} PC, {CORES_MAX} core")
print(f"Jumlah partisi          : {mentah_uji.getNumPartitions()} (minPartitions={MIN_PARTISI})")
print(f"Elemen/partisi (mentah) : min {min(glom_mentah):,} | maks {max(glom_mentah):,} | rata2 {st.mean(glom_mentah):,.0f}")
print(f"Tuple valid (uji/asli)  : {n_valid:,} / {n_asli:,}")
print(f"Accumulator file uji    : {hasil_acc}")
print(f"Accumulator file asli   : {nilai(acc_asli)}")
print(f"Spark UI                : {sc.uiWebUrl}  (Jobs berlabel 2a-...)")

## Sel 9 (opsional). Demo monitor task per Worker
Memakai pola `report()` dan `pantau()` dari materi **RDD-Cluster.ipynb**: setiap partisi (= task) melapor **RUNNING** lalu **DONE**
ke server laporan di **Soal2_Master_Monitor.ipynb** (port 8090), sehingga terlihat langsung PC Worker mana mengerjakan task mana.

Sebelum menjalankan sel ini:
1. Buka **Soal2_Master_Monitor.ipynb** di tab lain JupyterLab PC Master, jalankan Sel 1 (server) lalu Sel 3 (monitor).
2. Port **8090** di PC Master harus bisa dijangkau PC Worker (cek dari PC Worker: `Test-NetConnection 192.168.1.15 -Port 8090`).

**Jangan** memakai `pantau()` pada pengukuran waktu (2c, 2d): fungsi ini sengaja menambah jeda di setiap task.

In [ ]:
# ===== SEL 9 (opsional): Demo monitor task per Worker =====
import uuid, urllib.request

REPORT_URL = f"http://{MASTER_IP}:8090"     # server laporan di Soal2_Master_Monitor.ipynb

def report(run, app, nomor, status):
    """Dijalankan di executor: kirim status task ke monitor."""
    import json, os, socket, urllib.request
    data = {"run": run, "app": app, "task": nomor, "worker": socket.gethostname(),
            "pid": os.getpid(), "status": status}
    req = urllib.request.Request(REPORT_URL, data=json.dumps(data).encode("utf-8"),
                                 headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            return r.status == 200
    except Exception as e:
        print(f"Gagal melapor ke monitor: {e}")      # muncul di log executor
        return False

def pantau(rdd, label, jeda=2):
    """Bungkus RDD agar tiap partisi melapor RUNNING lalu DONE (pola materi RDD-Cluster)."""
    run = str(uuid.uuid4())
    def proses_partition(idx, isi):
        import time
        report(run, label, idx, "RUNNING")
        data = list(isi)
        time.sleep(jeda)                 # agar RUNNING sempat terlihat di monitor
        report(run, label, idx, "DONE")
        return iter(data)
    return rdd.mapPartitionsWithIndex(proses_partition)

try:
    with urllib.request.urlopen(REPORT_URL, timeout=3) as r:
        print("Monitor:", r.read().decode().strip())
except Exception as e:
    print("PERINGATAN: monitor tidak terjangkau dari driver ->", e)

acc_demo = buat_accumulator()
demo = pantau(mentah_uji.flatMap(buat_parser(acc_demo)), "2a-parse snapshot uji (32 task)")
tandai("2a-monitor Demo monitor task per Worker")
t0 = time.time()
print(f"Tuple valid: {demo.count():,} | {time.time() - t0:.1f} dtk (termasuk jeda pantau)")

# Sebaran task per PC dari sisi Spark (tanpa monitor): partisi -> PC yang mengerjakannya
def siapa(idx, isi):
    import socket
    yield (socket.gethostname(), 1)
tandai("2a-monitor Sebaran task per PC")
print("Task per PC:", dict(sc.parallelize(range(MIN_PARTISI), MIN_PARTISI)
                          .mapPartitionsWithIndex(siapa).reduceByKey(lambda a, b: a + b).collect()))